# Exercício 3 — Comparação entre GEPHI e NetLogo

Com base na sua experiência com os dois programas nos Exercícios 1 e 2, responda:

> **Qual a diferença de finalidade e uso dos dois programas (GEPHI e NetLogo) para o estudo de redes "artificiais" e "reais"?**

Considere:
- Para que tipo de análise cada programa é mais adequado?
- Que tipo de redes cada um consegue manipular (reais vs. artificiais)?
- Qual o nível de interatividade e aprendizado que cada um proporciona?

In [7]:
resposta_pergunta_3 = '''
Escreva sua resposta aqui.
'''

## Gerar resposta com IA

Execute a célula abaixo para que a IA analise os dados e gere uma resposta automaticamente.
Você pode editar o resultado depois se quiser.

In [8]:
from dotenv import load_dotenv
load_dotenv()
import os, json
from urllib.request import Request, urlopen

MODELO = os.getenv('OLLAMA_MODEL', 'ministral-3:3b')
BASE_URL = os.getenv('OLLAMA_BASE_URL', 'https://api.ollama.com').rstrip('/')
API_KEY = os.getenv('OLLAMA_API_KEY')

prompt = '''Voce e um assistente de um curso de sistemas de muitos agentes.
Responda em portugues, de forma clara e objetiva (maximo 4 paragrafos).

Pergunta: Qual a diferenca de finalidade e uso dos dois programas (GEPHI e NetLogo)
para o estudo de redes "artificiais" e "reais"?

Contexto:
- GEPHI: software desktop para visualizacao e analise de redes. Permite importar dados
  reais (CSV), aplicar layouts (ForceAtlas 2), colorir por atributo, calcular metricas
  (grau, betweenness, modularidade) e exportar grafos publicaveis.
- NetLogo: plataforma de simulacao baseada em agentes. Permite gerar redes artificials
  com algoritmos (Barabasi-Albert, Erdos-Renyi, small world, grade 2D), simular
  dinamicas sobre a rede, e explorar metricas interativamente.
- Redes reais: dados observados do mundo (ex.: relacoes entre alunos).
- Redes artificials: geradas por regras matematicas ou computacionais.'''

headers = {
    'Content-Type': 'application/json',
    'User-Agent': 'Mozilla/5.0',
    'Accept': 'application/json',
}
if API_KEY:
    headers['Authorization'] = f'Bearer {API_KEY}'

payload = {
    'model': MODELO,
    'messages': [{'role': 'user', 'content': prompt}],
    'stream': False,
    'options': {'temperature': 0.3, 'top_p': 0.9, 'num_predict': 800},
}

try:
    req = Request(
        BASE_URL + '/api/chat',
        data=json.dumps(payload).encode('utf-8'),
        headers=headers,
        method='POST',
    )
    with urlopen(req, timeout=60) as resp:
        dados = json.loads(resp.read().decode('utf-8'))
    resposta = dados.get('message', {}).get('content', '').strip()
    if resposta:
        resposta_pergunta_3 = resposta
        print('Resposta gerada com sucesso!')
    else:
        print('IA retornou resposta vazia.')
except Exception as e:
    print(f'Erro ao consultar IA: {type(e).__name__}: {e}')

print()
print('Resposta atual:')
print(resposta_pergunta_3)

Resposta gerada com sucesso!

Resposta atual:
**GEPHI** é principalmente um **ferramenta analítica e visual** para redes **reais**, focada na exploração de dados já existentes. Sua principal finalidade é ajudar na interpretação de estruturas complexas, como redes sociais, cibernéticas ou de transporte, através de visualizações interativas (como layouts baseados em força) e cálculos de métricas estatísticas (entre outros). É ideal para identificar padrões, modularidade ou centralidade em redes observadas, mas não permite gerar redes a partir de modelos teóricos. Sua força está na flexibilidade para manipular dados importados (CSV) e exportar resultados para publicação ou relatórios.

**NetLogo**, por outro lado, é uma **plataforma de simulação de agentes** voltada para redes **artificiais**, permitindo criar modelos dinâmicos e replicáveis. Ao contrário de GEPHI, ele não trabalha com dados reais, mas sim com algoritmos que produzem redes seguindo princípios teóricos (como o *Barabási-Al

## Exportar para o Moodle

In [9]:
import os
import shutil
import subprocess
from datetime import datetime

_repo = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_repo, 'setup.sh')):
        break
    _repo = os.path.dirname(_repo)
PASTA_MOODLE = os.path.join(_repo, 'work', 'tarefas_moodle')
os.makedirs(PASTA_MOODLE, exist_ok=True)
PASTA_AULA = os.path.join(PASTA_MOODLE, 'tarefa_aula05')
os.makedirs(PASTA_AULA, exist_ok=True)

agora = datetime.now().strftime('%Y-%m-%d %H:%M')

PATH_MD = os.path.join(PASTA_AULA, 'aula_05_exercicio_03.md')
with open(PATH_MD, 'w', encoding='utf-8') as f:
    f.write('# Aula 05 — Exercicio 03: Comparacao entre GEPHI e NetLogo\n\n')
    f.write(f'_Executado em: {agora}_\n\n')
    f.write('---\n\n')
    f.write('## Resposta do aluno\n\n')
    f.write(resposta_pergunta_3.strip())
    f.write('\n')

PATH_PDF = PATH_MD.replace('.md', '.pdf')
if shutil.which('pandoc'):
    subprocess.run(['pandoc', PATH_MD, '-o', PATH_PDF, '--pdf-engine=xelatex', '--resource-path', PASTA_AULA, '-V', 'mainfont=DejaVu Serif'], check=True)
else:
    print('Aviso: pandoc nao instalado. Execute ./setup.sh para instalar.')
print(f'Markdown: {os.path.abspath(PATH_MD)}')
print(f'PDF:      {os.path.abspath(PATH_PDF)}')
# Mesclar PDFs individuais em um unico PDF
if shutil.which('pdfunite'):
    import glob
    pdfs = sorted(glob.glob(os.path.join(PASTA_AULA, 'aula_05_exercicio_??.pdf')))
    if len(pdfs) == 3:
        PASTA_RESPOSTAS = os.path.join(PASTA_MOODLE, 'respostas')
        os.makedirs(PASTA_RESPOSTAS, exist_ok=True)
        PATH_COMBINADO = os.path.join(PASTA_RESPOSTAS, 'aula_05_completo.pdf')
        subprocess.run(['pdfunite'] + pdfs + [PATH_COMBINADO], check=True)
        print(f'PDF combinado: {os.path.abspath(PATH_COMBINADO)}')
    else:
        print(f'Aviso: encontrados {len(pdfs)} PDFs, esperados 3. Execute todos os exercicios primeiro.')


Markdown: /l/disk0/viniciusd/Projetos/simserv_jupyter_env/work/tarefas_moodle/aula_05_exercicio_03.md
PDF:      /l/disk0/viniciusd/Projetos/simserv_jupyter_env/work/tarefas_moodle/aula_05_exercicio_03.pdf
PDF combinado: /l/disk0/viniciusd/Projetos/simserv_jupyter_env/work/tarefas_moodle/respostas/aula_05_completo.pdf
